# 02A — Correction  ·  line A (NOAA)

Raw cubes in, corrected cubes out. **This notebook applies corrections and nothing else** —
it builds no regions and computes no metrics.

| in  | `data/raw/NOAA_<noaa>_<date>/region_01_{continuum,magnetogram,dopplergram}_cube.fits` + the per-frame files under `region_01/` |
|---|---|
| out | `data/processed/NOAA_<noaa>_<date>/region_01_continuum_cube.fits` — limb-darkening corrected, still DN/s |
|     | `…_magnetogram_corrected_cube.fits` — quiet-sun plane removed |
|     | `…_dopplergram_calibrated_cube.fits` — absolute LOS velocity, m/s |
|     | `…_frames.fits` — per-frame correction diagnostics |

**No mask file, deliberately.** Umbra, penumbra, hot spot and the quiet sun are all built in
`03A` from these cubes, every time it runs. The corrections re-read every per-frame FITS
header (~90 s per region) and none of that work depends on where an umbra boundary is drawn,
so keeping the regions out of here means retuning a threshold costs one `03A` run instead of
a full re-correction.

Caveats worth remembering when reading the output:

- the magnetogram is the **line-of-sight** field, uncorrected for `cos θ` — it is not `|B|`
- the continuum stays in **DN/s**; the DN→cgs factor is not applied, because everything
  downstream is a ratio of intensities and a global scale only makes the numbers harder to
  compare against the raw frames and against DS9
- a **gap** is a slot where a series genuinely has no frame. It stays NaN. Every step here
  is NaN-safe, and `region_01_frames.fits` records which slots they are.

The pipeline itself lives in `src/processing.py` (`load_aligned`, `remove_quiet_sun_plane`,
`process_region`, `write_region`) — it used to be defined in this notebook's cells, where
nothing else could reuse or test it.

In [ ]:
import pathlib
import sys

project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

from src import config, processing

# Every correction setting lives in src/config.PROCESSING, so what is written to disk is
# recorded in one place rather than in a notebook global. Print it, don't retype it.
print('Correction settings (src/config.py, PROCESSING):')
for key, value in config.PROCESSING.items():
    print(f'  {key:24s} = {value!r}')

## Which regions

Any directory under `data/raw/` that has a continuum cube built by `01A`.

In [ ]:
regions = sorted(p for p in config.RAW_DIR.glob('NOAA_*')
                 if (p / 'region_01_continuum_cube.fits').exists())

print(f'{len(regions)} region(s) with raw cubes:')
for i, region_dir in enumerate(regions):
    noaa = config.region_noaa(region_dir)
    crop = config.view_params(config.params_for(region_dir))['crop_to_data']
    done = (config.PROCESSED_DIR / region_dir.name / 'region_01_frames.fits').exists()
    print(f'  [{i}] {region_dir.name:26s} crop={crop!s:5s} '
          f'{"already processed" if done else ""}')
if not regions:
    print('  none — run 01A_download_data.ipynb first')

## Run

One loop over every region, rather than the `regions = regions[-1:]` slice this notebook
used to carry — that pinned it to whichever region was discovered last and made a full run
mean hand-editing the cell each time.

`SKIP_EXISTING = True` leaves already-processed regions alone, so re-running after adding
one AR costs only that AR. Set it to `False` after changing anything in
`config.PROCESSING`, because then every cube on disk is stale.

`crop_to_data` comes from `config.REGION_PARAMS`. It is a repair for a bad download, not
part of the pipeline: NOAA 11117's box shrinks partway through the window *and* its
magnetogram came down under a different box again (402×402 vs 433×433), so its three cubes
cannot be indexed against each other without trimming to their common data window. The real
fix is to re-download that region with one consistent box.

In [ ]:
SKIP_EXISTING = True

results = {}
for region_dir in regions:
    out_marker = config.PROCESSED_DIR / region_dir.name / 'region_01_frames.fits'
    if SKIP_EXISTING and out_marker.exists():
        print(f'{region_dir.name}: already processed — skipping')
        continue

    params = config.view_params(config.params_for(region_dir))
    print(f'{region_dir.name}:')
    result = processing.process_region(region_dir,
                                       crop_to_data=params['crop_to_data'],
                                       **config.PROCESSING)
    out_dir, written = processing.write_region(region_dir, result)
    results[region_dir.name] = result
    print(f'  -> {out_dir}')
    for label, path in written.items():
        print(f'     {label:12s} {pathlib.Path(path).name}')
    print()

## Diagnostics — did each correction do something sane?

These say how big each correction was, which is the only way to tell a correction that went
wrong from solar signal.

- **`I_qs`** — the frame's quiet-sun continuum. Should drift slowly, not jump.
- **`<C>`** — the mean limb-darkening factor, bounded in [0.368, 1]. It should *change*
  across the window as the region rotates. **If it comes back flat, the per-frame headers
  are not being read and the correction is wrong**, since one `C` map applied cube-wide
  would leave most of the effect in place and inject a spurious trend of its own.
- **Doppler terms** — `v_SDO` is by far the largest (of order km/s, and diurnal, which is
  exactly the period this project is measuring); `v_LSF` runs to a few hundred m/s and
  changes sign across the disk; `v_CLV` is a few hundred m/s; `v_gravity` is the constant
  636 m/s.
- **magnetogram plane** — `|grad|` is how much field the fitted plane spans across the box.
  A large gradient means the quiet-sun reference really was tilted; near zero means the
  plane fit had nothing to remove.

In [ ]:
for name, result in results.items():
    processing.summarize(name, result)
    print()

if not results:
    print('Nothing processed in this run — set SKIP_EXISTING = False to re-run a region.')

## What comes next

`03A_data_analysis.ipynb` reads the cubes written above, builds the regions, and does the
per-region analysis. Nothing here needs re-running when a threshold changes there.